# LBO-style sources & uses

High-level **leveraged buyout** mechanics: sources/uses, entry multiple, layered debt, amortization, and a simple **equity IRR / MOIC** grid.

## Concept

We approximate an LBO with explicit **debt balances**, **mandatory amortization**, and an **operating forecast**. Exit equity is distributable value after repaying remaining debt. Sensitivity on **entry vs exit EV/EBITDA** is a two-way table built by re-evaluating variants.

In [ ]:
from finstack.statements import ModelBuilder, Evaluator, FinancialModelSpec
from finstack.statements_analytics import run_sensitivity, evaluate_scenario_set, goal_seek, trace_dependencies, explain_formula, run_variance, evaluate_dcf, run_corporate_analysis, pl_summary_report, credit_assessment_report
import json
print("Loaded finstack.statements + finstack.statements_analytics")

from finstack.statements_analytics import run_sensitivity, goal_seek

PERIODS = ["2025", "2026", "2027", "2028", "2029"]
def s(v):
    return list(zip(PERIODS, v))

b = ModelBuilder("lbo-demo")
b.periods("2025..2029", "2025")

# Operating model
b.value("revenue", s([100.0, 105.0, 110.0, 115.0, 120.0]))
b.compute("ebitda", "revenue * 0.22")
b.value("capex", s([5.0, 5.0, 5.5, 5.5, 6.0]))
b.value("delta_nwc", s([1.0, 1.0, 1.0, 1.0, 1.0]))
b.compute("ufcf", "ebitda - capex - delta_nwc")

# Transaction — entry
entry_ebitda = 22.0
entry_multiple = 8.5
ev_entry = entry_ebitda * entry_multiple
fees = 3.0
equity_check = 75.0
tl_a = 90.0
revolver = 10.0
mezz = 15.0
sources_total = equity_check + tl_a + revolver + mezz
uses_total = ev_entry + fees
sources = sources_total
uses = uses_total
assert abs(sources - uses) < 0.01
b.value("sources_total_check", s([sources_total, 0.0, 0.0, 0.0, 0.0]))
b.value("uses_total_check", s([uses_total, 0.0, 0.0, 0.0, 0.0]))
b.compute("sources_uses_balance", "sources_total_check - uses_total_check")

# Debt schedules (simplified straight-line principal)
b.value("tl_a_balance", s([tl_a, 72.0, 54.0, 36.0, 18.0]))
b.value("revolver_balance", s([revolver, 8.0, 6.0, 4.0, 2.0]))
b.value("mezz_balance", s([mezz, 15.0, 15.0, 15.0, 15.0]))
b.value("tl_rate", s([0.06, 0.06, 0.06, 0.06, 0.06]))
b.value("rev_rate", s([0.08, 0.08, 0.08, 0.08, 0.08]))
b.value("mezz_rate", s([0.12, 0.12, 0.12, 0.12, 0.12]))
b.compute("tl_interest", "tl_a_balance * tl_rate")
b.compute("rev_interest", "revolver_balance * rev_rate")
b.compute("mezz_interest", "mezz_balance * mezz_rate")
b.compute("total_cash_interest", "tl_interest + rev_interest + mezz_interest")

# Exit (year 5): tie proceeds to terminal revenue so goal_seek / sensitivities work
exit_multiple = 9.5
b.value("exit_multiple_scalar", s([0.0, 0.0, 0.0, 0.0, exit_multiple]))
b.compute("total_debt", "tl_a_balance + revolver_balance + mezz_balance")
b.compute("equity_proceeds", "revenue * 0.22 * exit_multiple_scalar - total_debt")

spec = b.build()
res = Evaluator().evaluate(spec)
model_json = spec.to_json()

ep = res.get("equity_proceeds", "2029")
moic = (ep / equity_check) if ep is not None else float("nan")
exit_ev = (120.0 * 0.22) * exit_multiple
print("Sources & uses balanced (Y2025):", res.get("sources_uses_balance", "2025"))
print("Entry EV:", ev_entry, "| Equity:", equity_check)
print("Exit EV (Y2029):", exit_ev, "| Total debt at exit:", res.get("total_debt", "2029"))
print("Equity proceeds (approx):", ep, "| MOIC:", round(moic, 3))

# Simple IRR on equity cash flows (t0 -equity, t1–t4 hold, t5 exit = 5-year horizon)
flows = [-equity_check, 0.0, 0.0, 0.0, 0.0, ep if ep is not None else 0.0]


def npv(rate, cfs):
    return sum(cf / ((1.0 + rate) ** i) for i, cf in enumerate(cfs))


lo, hi = -0.9, 3.0
for _ in range(80):
    mid = (lo + hi) / 2.0
    if npv(mid, flows) > 0:
        lo = mid
    else:
        hi = mid
irr = (lo + hi) / 2.0
print("Equity IRR (approx, annual):", round(irr, 4))

sens_cfg = json.dumps(
    {
        "mode": "Diagonal",
        "parameters": [
            {
                "node_id": "revenue",
                "period_id": "2029",
                "base_value": 120.0,
                "perturbations": [-10.0, 0.0, 10.0],
            }
        ],
        "target_metrics": ["equity_proceeds", "ufcf"],
    }
)
print("--- run_sensitivity (revenue@2029 vs targets) ---")
print(run_sensitivity(model_json, sens_cfg))


In [ ]:
print("Two-way LBO grid (uses prior cell variables)")
# Two-way sensitivity: entry multiple vs exit multiple (re-build EV markers)

def build_lbo_spec(entry_m, exit_m):
    bb = ModelBuilder("lbo-grid")
    bb.periods("2025..2029", "2025")
    bb.value("revenue", s([100.0, 105.0, 110.0, 115.0, 120.0]))
    bb.compute("ebitda", "revenue * 0.22")
    # Higher entry multiple -> more leverage in this stylized grid (vs 8.5x baseline)
    debt_scale = entry_m / 8.5
    tl0 = tl_a * debt_scale
    rev0 = revolver * debt_scale
    mz0 = mezz * debt_scale
    bb.value("capex", s([5.0, 5.0, 5.5, 5.5, 6.0]))
    bb.value("delta_nwc", s([1.0, 1.0, 1.0, 1.0, 1.0]))
    bb.compute("ufcf", "ebitda - capex - delta_nwc")
    bb.value("tl_a_balance", s([tl0, tl0 * 0.8, tl0 * 0.6, tl0 * 0.4, tl0 * 0.2]))
    bb.value("revolver_balance", s([rev0, rev0 * 0.8, rev0 * 0.6, rev0 * 0.4, rev0 * 0.2]))
    bb.value("mezz_balance", s([mz0, mz0, mz0, mz0, mz0]))
    bb.value("tl_rate", s([0.06, 0.06, 0.06, 0.06, 0.06]))
    bb.value("rev_rate", s([0.08, 0.08, 0.08, 0.08, 0.08]))
    bb.value("mezz_rate", s([0.12, 0.12, 0.12, 0.12, 0.12]))
    bb.compute("tl_interest", "tl_a_balance * tl_rate")
    bb.compute("rev_interest", "revolver_balance * rev_rate")
    bb.compute("mezz_interest", "mezz_balance * mezz_rate")
    bb.value("exit_multiple_scalar", s([0.0, 0.0, 0.0, 0.0, exit_m]))
    bb.compute("total_debt", "tl_a_balance + revolver_balance + mezz_balance")
    bb.compute("equity_proceeds", "revenue * 0.22 * exit_multiple_scalar - total_debt")
    return bb.build()


print("Entry multiple x Exit multiple -> MOIC (equity check fixed)")
for em in (7.5, 8.5, 9.5):
    row = []
    for xm in (8.0, 9.5, 11.0):
        r = Evaluator().evaluate(build_lbo_spec(em, xm))
        ep2 = r.get("equity_proceeds", "2029")
        row.append(round((ep2 / equity_check) if ep2 is not None else float("nan"), 3))
    print(em, row)

try:
    solved, mj = goal_seek(
        model_json,
        target_node="equity_proceeds",
        target_period="2029",
        target_value=120.0,
        driver_node="revenue",
        driver_period="2029",
        update_model=False,
    )
    print("goal_seek revenue@2029 for equity_proceeds=120:", solved)
except Exception as e:
    print("goal_seek (optional):", e)


## Takeaways

- **Sources & uses** tie deal proceeds to purchase price and fees; the model checks closure with a dedicated balance node.
- **Tranche-level interest** feeds cash needs; principal schedules drive **exit total debt** and **sponsor proceeds**.
- **MOIC / IRR** are standard sponsor metrics; `goal_seek` and `run_sensitivity` connect operating drivers to exit proceeds.